# 🌲 Step 5 — Carbon Stock & Uncertainty
**Forest Carbon Continuity Pipeline · Sentinel-1/2 Hybrid for EU LULUCF Compliance**

## What this step does
Takes a species map (synthetic here, Step 4's output in production) and converts it into
**biomass with an explicit ±95% confidence interval**, then into carbon tonnes and CO₂-equivalent
— the numbers an EU LULUCF submission actually reports. Outputs: a 2-band GeoTIFF
(biomass, uncertainty) and a per-species summary table.

## The Science
Ireland reports with **Tier 2** factors: a Species × Age-Class matrix instead of generic IPCC
defaults. This prototype uses the mature-stock peaks from `configs/ireland_config.yaml` logic —
Sitka Spruce 450 t/ha, Lodgepole Pine 380 t/ha, Native Mixed 300 t/ha — realised at **80% of
peak** to represent the standing stock of a working plantation. Each species carries an
uncertainty factor (15% / 18% / 25%), propagated as

$$CI_{95} = \pm 1.96 \times \sigma$$

Biomass-to-carbon uses the standard 50% carbon fraction, and CO₂e multiplies by 3.67 (44/12).
Canopy density is deliberately **not** a predictor here: Irish plantations are even-aged
monocultures, so age+species predicts stock far better than canopy cover (see Step 6).

In [ ]:
# ============================================================
# Environment detection & package setup (runs first, auto-detects
# Google Colab vs local Jupyter - nothing needs manual pip install)
# ============================================================
import sys, subprocess

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

_ensure("numpy"); _ensure("pandas"); _ensure("matplotlib")
_ensure("rasterio"); _ensure("geopandas"); _ensure("scikit-learn")
_ensure("openpyxl"); _ensure("ipywidgets")

def outpath(name):
    """Returns a working save path for the current environment."""
    return f"/content/{name}" if IN_COLAB else name

print(f"Environment detected: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

# ============================================================
# Unified cross-environment helpers:
# download buttons, regenerate buttons, upload widgets
# ============================================================
import zipfile
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

_generated_files = {}  # friendly label -> filepath, used by the final "Download All" bundle

def register_file(label, path):
    if label:
        _generated_files[label] = path

def make_download_button(path, label=None, register_as=None):
    """Cross-environment download button. Colab: real browser download.
    Local Jupyter: clickable FileLink. Reads the file fresh at click time."""
    label = label or f"Download {path}"
    if register_as:
        register_file(register_as, path)
    btn = widgets.Button(description=label, button_style="info", icon="download")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            if IN_COLAB:
                colab_files.download(path)
            else:
                display(FileLink(path))
                print(f"Click the link above to download '{path}'.")

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_regenerate_button(rerun_fn, label="🔄 Regenerate & Re-run"):
    """Button that calls rerun_fn() (fresh random seed, full re-run of this
    section) and shows the refreshed output inline below the button."""
    btn = widgets.Button(description=label, button_style="warning", icon="refresh")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            rerun_fn()

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_upload_widget(on_loaded, accept=".csv,.xlsx", label="Upload replacement file"):
    """Colab: native blocking file picker via google.colab.files.upload().
    Local Jupyter: ipywidgets.FileUpload + Load button (non-blocking)."""
    out = widgets.Output()

    if IN_COLAB:
        btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                print("Choose a file in the dialog, or Cancel to keep current data.")
                uploaded = colab_files.upload()
                if uploaded:
                    fname = next(iter(uploaded))
                    on_loaded(fname, uploaded[fname])
                else:
                    print("No file selected - keeping current data.")

        btn.on_click(on_click)
        display(widgets.VBox([btn, out]))
    else:
        uploader = widgets.FileUpload(accept=accept, multiple=False, description="Choose file")
        load_btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                val = uploader.value
                if not val:
                    print("No file selected yet - choose a file above, then click the button.")
                    return
                item = list(val.values())[0] if isinstance(val, dict) else val[0]
                if isinstance(item, dict) and "metadata" in item:
                    fname = item["metadata"]["name"]     # ipywidgets 7.x shape
                else:
                    fname = item["name"]                  # ipywidgets 8.x shape
                on_loaded(fname, bytes(item["content"]))

        load_btn.on_click(on_click)
        display(widgets.VBox([uploader, load_btn, out]))

def make_download_all_button(label="⬇️ Download ALL outputs (.zip)"):
    """Zips every output registered so far and offers it as one download."""
    zip_path = outpath("outputs_bundle.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for _label, _path in _generated_files.items():
            try:
                zf.write(_path)
            except FileNotFoundError:
                pass
    make_download_button(zip_path, label=label)

print("Helpers ready: make_download_button / make_regenerate_button / make_upload_widget")

In [ ]:
# ============================================================
# Standalone synthetic species map (Day-1 ready)
# ============================================================
import numpy as np
import rasterio
from rasterio.transform import from_bounds

H, W = 120, 120
BOUNDS = (-6.4, 52.95, -6.1, 53.15)
TRANSFORM = from_bounds(*BOUNDS, W, H)
PIXEL_AREA_HA = 100.0 / 10000.0        # 10 m pixels -> 0.01 ha
BIOMASS_PEAK = {1: 450.0, 2: 380.0, 3: 300.0}        # t/ha mature peaks
UNCERTAINTY_FACTOR = {1: 0.15, 2: 0.18, 3: 0.25}      # species-specific sigma factors
SPECIES_NAMES = {1: "Sitka Spruce", 2: "Lodgepole Pine", 3: "Native Mixed"}

def generate_species_map(seed=5):
    """Synthetic species map - same blob logic as Step 4."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:H, 0:W]

    def blob(cx, cy, sx, sy):
        return np.exp(-(((xx - cx) / sx) ** 2 + ((yy - cy) / sy) ** 2))

    zones = np.stack([blob(35, 40, 20, 16), blob(85, 85, 24, 20), blob(80, 25, 18, 18)])
    return (np.argmax(zones, axis=0) + 1).astype("uint8")

species = generate_species_map()
print(f"Species map: {H}x{W} pixels @ {PIXEL_AREA_HA} ha each "
      f"({H*W*PIXEL_AREA_HA:.0f} ha total)")

In [ ]:
# ============================================================
# Run Tier-2 carbon accounting with uncertainty propagation
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

def run_carbon(species, tag="synthetic"):
    biomass = np.zeros_like(species, dtype="float32")
    uncertainty = np.zeros_like(species, dtype="float32")
    for sp, peak in BIOMASS_PEAK.items():
        m = species == sp
        biomass[m] = peak * 0.8                                        # Tier-2 standing stock
        uncertainty[m] = biomass[m] * UNCERTAINTY_FACTOR[sp] * 1.96    # +/-95% CI

    rows = []
    for sp in [1, 2, 3]:
        m = species == sp
        area_ha = int(m.sum()) * PIXEL_AREA_HA
        bio_t = float(biomass[m].mean()) * area_ha
        ci_t = float(uncertainty[m].mean()) * area_ha
        rows.append({"Species": SPECIES_NAMES[sp], "Area (ha)": round(area_ha, 1),
                     "Biomass (t)": round(bio_t, 1), "CI95 (+/- t)": round(ci_t, 1),
                     "Carbon (tC)": round(bio_t * 0.5, 1),
                     "CO2e (t)": round(bio_t * 0.5 * 3.67, 1)})
    summary = pd.DataFrame(rows)
    totals = {"Species": "TOTAL", "Area (ha)": summary["Area (ha)"].sum(),
              "Biomass (t)": summary["Biomass (t)"].sum(),
              "CI95 (+/- t)": float(np.sqrt((summary["CI95 (+/- t)"]**2).sum())),
              "Carbon (tC)": summary["Carbon (tC)"].sum(),
              "CO2e (t)": summary["CO2e (t)"].sum()}
    summary = pd.concat([summary, pd.DataFrame([totals])], ignore_index=True)
    print(summary.to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    im0 = axes[0].imshow(biomass, cmap="YlOrRd", vmin=0, vmax=400)
    axes[0].set_title("Biomass (t/ha)"); plt.colorbar(im0, ax=axes[0], fraction=0.046)
    im1 = axes[1].imshow(uncertainty, cmap="Purples", vmin=0, vmax=120)
    axes[1].set_title("Uncertainty (+/-95% CI, t/ha)"); plt.colorbar(im1, ax=axes[1], fraction=0.046)
    vals = summary[summary["Species"] != "TOTAL"]
    axes[2].bar(vals["Species"], vals["Biomass (t)"], yerr=vals["CI95 (+/- t)"],
                color=["#2e7d32", "#f9a825", "#6d4c41"], capsize=5)
    axes[2].set_title("Total biomass per species (+/-95% CI)")
    axes[2].set_ylabel("Biomass (t)")
    plt.tight_layout()

    chart_path = outpath("step5_carbon_chart.png")
    fig.savefig(chart_path, dpi=120, bbox_inches="tight")
    plt.show()

    tif_path = outpath("step5_biomass_uncertainty.tif")
    with rasterio.open(tif_path, "w", driver="GTiff", height=H, width=W, count=2,
                       dtype="float32", crs="EPSG:4326", transform=TRANSFORM) as dst:
        dst.write(biomass, 1)
        dst.write(uncertainty, 2)
        dst.set_band_description(1, "biomass_t_per_ha")
        dst.set_band_description(2, "ci95_t_per_ha")

    xlsx_path = outpath("step5_carbon_summary.xlsx")
    summary.to_excel(xlsx_path, index=False)

    make_download_button(chart_path, label="⬇️ Download carbon chart (.png)",
                         register_as=f"Step 5 chart ({tag})")
    make_download_button(tif_path, label="⬇️ Download biomass+CI GeoTIFF (.tif)",
                         register_as=f"Step 5 biomass raster ({tag})")
    make_download_button(xlsx_path, label="⬇️ Download carbon summary (.xlsx)",
                         register_as=f"Step 5 summary ({tag})")
    return summary

summary = run_carbon(species)

**Interpretation:** the left panel is the headline product — tonnes of biomass per hectare,
color-coded by species zone. The middle panel is the honest part: Native Mixed carries the
widest error bar because its allometry is least studied (25% uncertainty factor vs 15% for
Sitka Spruce). The right panel shows why species mapping (Step 4) matters at all: shift a block
of land from "Native Mixed" to "Sitka Spruce" and the same hectares gain a third more carbon.
The TOTAL row combines uncertainties in quadrature — correlated errors would widen it, which is
exactly what the uncertainty log documents in production.

In [ ]:
# ============================================================
# Optional: upload your own species map (.tif, uint8, values 1/2/3)
# ============================================================
def handle_species_upload(fname, content_bytes):
    tmp = outpath("uploaded_species_map.tif")
    open(tmp, "wb").write(content_bytes)
    try:
        with rasterio.open(tmp) as src:
            u_species = src.read(1)
    except Exception as e:
        print(f"❌ Could not read '{fname}': {e}")
        return
    if not np.isin(u_species, [1, 2, 3]).all():
        found = sorted(np.unique(u_species).tolist())
        print(f"❌ Upload rejected - species map must contain only 1/2/3, found {found}.")
        return
    print(f"✅ SUCCESS: loaded species map from '{fname}'. Re-running carbon accounting...")
    run_carbon(u_species.astype("uint8"), tag=f"uploaded:{fname}")

make_upload_widget(handle_species_upload, accept=".tif,.tiff", label="Upload species map (.tif)")
print("Skip to continue with the synthetic map.")

# ============================================================
# Regenerate control
# ============================================================
def rerun_step5(seed=None):
    """Fresh synthetic species map + full carbon re-run."""
    global species, summary
    seed = seed or np.random.randint(0, 10_000)
    species = generate_species_map(seed=seed)
    print(f"[seed={seed}] regenerated species map")
    summary = run_carbon(species, tag=f"seed-{seed}")

make_regenerate_button(rerun_step5, label="🔄 Regenerate species map & re-run accounting")
make_download_all_button()